In [1]:
!pip install kagglehub -q
import kagglehub, os, shutil, glob
path = kagglehub.dataset_download("emmarex/plantdisease")
print(path)

Using Colab cache for faster access to the 'plantdisease' dataset.
/kaggle/input/plantdisease


In [2]:
pv = os.path.join(path, "PlantVillage")
new_base = "/content/research_dataset"

classes = ["healthy", "surface", "deep", "aggressive"]

for c in classes:
    os.makedirs(os.path.join(new_base, c), exist_ok=True)

mapping = {
    "healthy": "Tomato_healthy",
    "surface": "Tomato_Early_blight",
    "deep": "Tomato_Late_blight",
    "aggressive": "Tomato_Tomato_YellowLeaf_Curl_Virus"
}

for target, source in mapping.items():
    src_folder = os.path.join(pv, source)
    files = glob.glob(src_folder+"/*.jpg") + glob.glob(src_folder+"/*.JPG")

    for f in files:
        shutil.copy(f, os.path.join(new_base, target, os.path.basename(f)))

print("Recovered dataset:", os.listdir(new_base))

Recovered dataset: ['surface', 'deep', 'healthy', 'aggressive']


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    new_base,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = image_dataset_from_directory(
    new_base,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

class_names = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print(class_names)

Found 4499 files belonging to 4 classes.
Using 3600 files for training.
Found 4499 files belonging to 4 classes.
Using 899 files for validation.
['aggressive', 'deep', 'healthy', 'surface']


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Small

# SE Block
def se_block(x, ratio=8):
    ch = int(x.shape[-1])

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(ch // ratio, activation="relu")(se)
    se = layers.Dense(ch, activation="sigmoid")(se)
    se = layers.Reshape((1,1,ch))(se)

    return layers.Multiply()([x, se])

# Backbone
base_se = MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)
base_se.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = layers.Rescaling(1./255)(inputs)
x = base_se(x, training=False)
x = se_block(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

se_model = Model(inputs, outputs)

se_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history3 = se_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

print("SE Model done.")

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 74s 576ms/step - accuracy: 0.4217 - loss: 1.0964 - val_accuracy: 0.5695 - val_loss: 1.0371
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 54s 477ms/step - accuracy: 0.5250 - loss: 1.0234 - val_accuracy: 0.5951 - val_loss: 0.9659
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 53s 471ms/step - accuracy: 0.5750 - loss: 0.9587 - val_accuracy: 0.6452 - val_loss: 0.9175
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 94s 576ms/step - accuracy: 0.5878 - loss: 0.9256 - val_accuracy: 0.5695 - val_loss: 0.8762
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 82s 576ms/step - accuracy: 0.5894 - loss: 0.8956 - val_accuracy: 0.5984 - val_loss: 0.8500
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 82s 571ms/step - accuracy: 0.5958 - loss: 0.8697 - val_accuracy: 0.6007 - val_loss: 0.8287
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 64s 562ms/step - accuracy: 0.6033 - loss: 0.8504 - val_accuracy: 0.6151 - val_loss: 0.8120
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Predictions
y_true = np.concatenate([y.numpy() for x, y in val_ds], axis=0)
y_pred = np.argmax(heca_model.predict(val_ds), axis=1)

cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])

# Normalized matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

# Figure
plt.figure(figsize=(16,12))

# 1 Accuracy
plt.subplot(2,2,1)
plt.plot(history.history["val_accuracy"], label="Baseline")
plt.plot(history2.history["val_accuracy"], label="HECA-CBAM")
plt.plot(history3.history["val_accuracy"], label="SE")
plt.title("Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# 2 Loss
plt.subplot(2,2,2)
plt.plot(history.history["val_loss"], label="Baseline")
plt.plot(history2.history["val_loss"], label="HECA-CBAM")
plt.plot(history3.history["val_loss"], label="SE")
plt.title("Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# 3 Heatmap
plt.subplot(2,2,3)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=False
)
plt.title("Confusion Heatmap")
plt.xlabel("Predicted")
plt.ylabel("True")

# 4 Inter-Class Difference
plt.subplot(2,2,4)
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="magma",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=False
)
plt.title("Inter-Class Difference")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()
plt.show()